Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Getting coordinates: geocoding

A point map needs a **latitude** and **longitude** for every point. But data usually comes with an **address** or a **postcode** instead. Turning these into coordinates is called **geocoding**.

We use two free APIs (Week 4), with loops (Week 5):

1. **[postcodes.io](https://postcodes.io)**: UK postcodes → coordinates, plus the council, constituency and more. No key needed.
2. **[Nominatim](https://nominatim.org)** (OpenStreetMap): place names and addresses, anywhere in the world. Free, but only **one request per second**.

Then we save the results and draw them on a map.

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet

In [ ]:
import requests        # fetch things from the internet (Week 4)
import time            # pause between requests (Week 5)
import pandas as pd    # tables
import altair as alt   # charts

<br><br>

---

<br>

## 1. One postcode

postcodes.io takes the postcode in the web address. The result is JSON, with the answer inside `result`:

In [ ]:
postcode = "WC2A 2AE"     # LSE

response = requests.get(f"https://api.postcodes.io/postcodes/{postcode}")
print(response.status_code)

result = response.json()["result"]
print(f"Latitude: {result['latitude']}, longitude: {result['longitude']}")
print(f"Council: {result['admin_district']} ({result['codes']['admin_district']})")

<br>

As well as coordinates, we get the postcode's **council** (and its code, e.g. `E09000033`), constituency, region and more. Look through everything it returns:

In [ ]:
result

<br>

> The council **code** is the same kind of code our map files use (`properties.id`). So postcodes can be joined to council-level data and maps with a lookup (see section 5).

<br><br>

---

<br>

## 2. Many postcodes, with a loop

The UK's parliaments and assemblies, and LSE. The pattern from Week 5: start an empty list, request each postcode, and **if** it worked, add a row.

In [ ]:
places = {
    "UK Parliament": "SW1A 0AA",
    "Senedd (Wales)": "CF99 1SN",
    "Scottish Parliament": "EH99 1SP",
    "Northern Ireland Assembly": "BT4 3XX",
    "LSE": "WC2A 2AE",
}

rows = []

for name, postcode in places.items():          # .items() gives each name and its postcode
    response = requests.get(f"https://api.postcodes.io/postcodes/{postcode}")

    if response.status_code == 200:
        result = response.json()["result"]
        rows.append({"name": name, "postcode": postcode,
                     "latitude": result["latitude"], "longitude": result["longitude"],
                     "council": result["admin_district"], "council_code": result["codes"]["admin_district"]})
        print(f"OK: {name}")
    else:
        print(f"Problem with {name} ({postcode}): error {response.status_code}")

    time.sleep(0.2)     # be polite to the API

points = pd.DataFrame(rows)
points

<br>

### Up to 100 at once

postcodes.io can also look up a whole list in **one** request. Instead of `get`, we `post` (send) the list as JSON. Each answer has the `query` (what we sent) and its `result` (or `None` if the postcode wasn't found):

In [ ]:
response = requests.post("https://api.postcodes.io/postcodes",
                         json={"postcodes": list(places.values())})

for answer in response.json()["result"]:
    if answer["result"] is not None:
        print(answer["query"], answer["result"]["latitude"], answer["result"]["longitude"])
    else:
        print(answer["query"], "not found")

<br><br>

---

<br>

## 3. Addresses and place names: Nominatim

Nominatim searches OpenStreetMap for any place or address. Its [usage policy](https://operations.osmfoundation.org/policies/nominatim/) asks for two things:
- a **`User-Agent`** header that says who you are (any short description of your project);
- no more than **one request per second**: `time.sleep(1)`.

The answer is a list of matches; we take the first, `[0]`. Its `lat` and `lon` are text, so we turn them into numbers with `float()`.

In [ ]:
headers = {"User-Agent": "PP434 course notebook (LSE student project)"}

searches = ["Cardiff Castle", "Edinburgh Castle", "Belfast City Hall", "Tower of London"]

found = []

for search in searches:
    response = requests.get("https://nominatim.openstreetmap.org/search",
                            params={"q": search, "format": "json", "limit": 1, "countrycodes": "gb"},
                            headers=headers)
    matches = response.json()

    if response.status_code == 200 and len(matches) > 0:
        best = matches[0]
        found.append({"name": search, "latitude": float(best["lat"]), "longitude": float(best["lon"])})
        print(f"OK: {search} → {best['display_name'][:60]}...")
    else:
        print(f"Not found: {search}")

    time.sleep(1)     # Nominatim allows one request per second

landmarks = pd.DataFrame(found)
landmarks

<br>

> **Always check the matches.** A search for "Cambridge" could find Cambridge, Massachusetts. `countrycodes="gb"` limits the search to the UK, and `display_name` shows what was found. Geocoding is a guess: check a few results on a map.

> **Be careful with personal data.** Don't geocode people's home addresses without permission, and never publish them.

<br><br>

---

<br>

## 4. Save and map

Join the two tables, save them as a CSV for your website, and draw them on a UK base map (as in the lecture):

In [ ]:
all_points = pd.concat([points[["name", "latitude", "longitude"]], landmarks])
all_points.to_csv("geocoded_places.csv", index=False)

uk = alt.topo_feature("https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/gbr/CTRY_UK.json", "geog")
base = alt.Chart(uk).mark_geoshape(fill="#e3e3e3", stroke="white")

dots = alt.Chart(all_points).mark_circle(size=80, color="#d35400").encode(
    longitude="longitude:Q",
    latitude="latitude:Q",
    tooltip=[alt.Tooltip("name:N").title("Place")],
)

(base + dots).project("mercator").properties(width=300, height=400, title="Geocoded places")

<br><br>

---

<br>

## 5. Bonus: from postcode to strategic authority

postcodes.io gave us each place's **council code**. map-data has a lookup table from council codes to regions and strategic authorities. `merge` joins two tables on a shared column, like a Vega-Lite lookup:

In [ ]:
lookup = pd.read_csv("https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/gbr/lookups/LAD_UK_2025_12.csv")

points.merge(lookup[["lad_id", "rgn_name", "sa_name"]], left_on="council_code", right_on="lad_id", how="left")

<br>

Only English councils belong to a strategic authority (and London's is the GLA), so the others are blank (`NaN`).

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1.** Make a list of five postcodes you know (your college, a workplace, a museum). Geocode them with the bulk request, and map them.

**2.** Change the Nominatim loop to search for five places outside the UK. (Hint: remove `countrycodes`.)

**3.** For your project: find a dataset with addresses or postcodes (e.g. schools, hospitals, or GP surgeries), geocode the first 20 rows, and make a point map. Remember `time.sleep()`.

In [ ]:
# Your code here
